# Trim Envelope

Calculate the set of trim points for an aircraft over a range of airspeeds
and range of flight path angles $\gamma$. The required thrust and AoA is
indicated via a colormap for each trim point.

In [ ]:
import sys

if 'google.colab' in sys.modules:
    print('Running on Google Colab – installing jsbsim …')
    !pip install jsbsim

PATH_TO_JSBSIM_FILES = None

## Initialize FDM

In [ ]:
import jsbsim

# --- JSBSim Initialization ---
# These lines initialize the flight dynamics model.

# Create a flight dynamics model (FDM) instance.
fdm = jsbsim.FGFDMExec(PATH_TO_JSBSIM_FILES)

fdm.set_debug_level(0) # Suppress verbose JSBSim console output

if fdm is not None:
    print("FDM created successfully")
    fdm.set_debug_level(0) # Suppress verbose JSBSim console output
else:
    print("Failed to create FDM")

## Tweak aircraft XML file: remove `<input/>` nodes from the officially released files

In [ ]:
import os
import xml.etree.ElementTree as ET

AIRCRAFT_NAME="737"

ac_xml_file_path = os.path.join(fdm.get_root_dir(), f'aircraft/{AIRCRAFT_NAME}/{AIRCRAFT_NAME}.xml')
print(f"Aircraft original XML file: {ac_xml_file_path}")

print("Parsing XML ...")
ac_xml_tree = ET.parse(ac_xml_file_path)

# Save a copy of the original XML file for backup
backup_xml_file_path = ac_xml_file_path.replace('.xml', '_backup.xml')
print(f"Saving backup XML file: {backup_xml_file_path}")
ac_xml_tree.write(backup_xml_file_path)

# Parse the XML and get the root element
ac_xml_root = ac_xml_tree.getroot()
# Remove <input ... /> occurrences with a 'port' attribute
for x in ac_xml_root.findall('input'):
    has_port = x.get('port') is not None
    if has_port:
        print(f"\tRemoving <input ... /> with port: {x.get('port')}")
        ac_xml_root.remove(x)

print(f"Saving modified XML: {ac_xml_file_path}")
ac_xml_tree.write(ac_xml_file_path)

#check that the input occurrences are removed
ac_xml_tree2 = ET.parse(ac_xml_file_path)
ac_xml_root2 = ac_xml_tree2.getroot()
inputs = ac_xml_root2.findall('input')
if not inputs:
    print("All <input ... /> occurrences successfully removed.")
else:
    print(f"Warning: Found {len(inputs)} <input/> occurrences remaining.")  

## Initialize the JSBSim executable

In [ ]:
import matplotlib.pyplot as plt

if fdm is not None:
    print("FDM instance found, proceeding with model loading...")
    # Load the aircraft model
    if fdm.load_model(AIRCRAFT_NAME):
        print(f"Aircraft model '{AIRCRAFT_NAME}' loaded successfully")
    else:
        print(f"Failed to load aircraft model '{AIRCRAFT_NAME}'")
else:
    print("FDM instance not found, cannot load aircraft model")

## Work on trim conditions

In [ ]:
import math

# Set engines running
fdm['propulsion/set-running'] = -1

# --- Simulation Parameters ---
# These lines set the simulation parameters.

# Set alpha range for trim solutions
fdm['aero/alpha-max-rad'] = math.radians(12)   # Maximum angle of attack in radians.
fdm['aero/alpha-min-rad'] = math.radians(-4.0) # Minimum angle of attack in radians.

# Set envelope limits
min_speed = 120   # Set the minimum calibrated airspeed (kts).
max_speed = 460   # Set the maximum calibrated airspeed (kts).
altitude = 15000  # Set the altitude (ft).
min_gamma = -10   # Set the minimum flight path angle (deg).
max_gamma = 10    # Set the maximum flight path angle (deg).

# Trim results
results = []  # Initialize an empty list to store the trim results.


# --- Simulation running ---
# These lines run the simulation for each conditions.

# Iterate over a range of speeds and for each speed a range of flight path angles (gamma)
# and check whether a trim point is possible
for speed in range(min_speed, max_speed, 10):
    for gamma in range(min_gamma, max_gamma, 1):
        # set the initial conditions
        fdm['ic/h-sl-ft'] = altitude  # altitude above sea level (ft)
        fdm['ic/vc-kts'] = speed      # calibrated airspeed (kts)
        fdm['ic/gamma-deg'] = gamma   # flight path angle (deg)

        # Initialize the aircraft with initial conditions
        fdm.run_ic()

        # Trim the aircraft.
        try:
            # 1 means straight flight by using all changeable control variables.
            fdm['simulation/do_simple_trim'] = 1

            # Append the airspeed, AoA, gamma, and throttle to the results list.
            results.append((fdm['velocities/vc-kts'], fdm['aero/alpha-deg'], gamma, fdm['fcs/throttle-cmd-norm[0]']))

        except jsbsim.TrimFailureError:
            pass  # Ignore trim failures


# --- Plot Results ---
# This section plots the simulation results.

# Extract the trim results
speed, alpha, gamma, throttle = zip(*results)

plt.rcParams["figure.figsize"] = (16, 8)  # Set the figure size for matplotlib plots.

# Plot the trim envelope results, with required thrust and AoA indicated via a colormap
fig, (axThrust, axAoA) = plt.subplots(1, 2)  # Create a figure with two subplots (thrust and AoA)

# Graph data for each of the sub plots (title, ax, data)
graph_data = [ ('Thrust', axThrust, throttle), ('AoA', axAoA, alpha) ]

for title, ax, data in graph_data:
    # Scatter plot with airspeed on x-axis, gamma on y-axis and either thrust setting or
    # AoA indicated via color map
    scatter = ax.scatter(speed, gamma, c=data, cmap='viridis')
    cb = fig.colorbar(scatter, ax=ax)  # Add a colorbar to the plot.
    cb.set_label(title)

    # Graph axis range for speed and gamma
    ax.set_xlim(min_speed - 20, max_speed + 20)
    ax.set_ylim(min_gamma * 2, max_gamma * 2)

    ax.grid(True, linestyle='-.')   # Add a grid to the plot.

    ax.set_xlabel('IAS (kt)')
    ax.set_ylabel('Flight Path Angle $\gamma$ (deg)')
    ax.set_title(f'Trim Envelope - {title}')

plt.show()  # Display the plot.